# Practical Class 3: The Geometry of Unsupervised Learning

Welcome to the final practical class of Phase 2! Today, we drop the training wheels. You have no target variable, no ground truth, and no labels. 

Before we dive into the Kaggle competition, we are going to spend 1 hour looking under the hood of clustering and dimensionality reduction. We need to understand *how* these algorithms view geometry so you know exactly which tool to use when.

---
## Part 1.1: Convex Clustering (K-Means & GMM)

Algorithms like K-Means and Gaussian Mixture Models (GMM) assume our clusters are **convex**—meaning if you draw a line between any two points in a cluster, that line stays entirely inside the cluster. Think of them as spheres or ovals.

Let's watch **K-Means** converge step-by-step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

sns.set_theme(style="whitegrid")

# 1. Generate easy, convex blobs
X_blobs, y_blobs = make_blobs(n_samples=300, centers=3, cluster_std=1.0, random_state=42)

# 2. Plot K-Means convergence step-by-step
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
iters = [1, 2, 3, 6]

for ax, i in zip(axes, iters):
    # We restrict max_iter to force it to stop early so we can see the steps
    km = KMeans(n_clusters=3, init='k-means++', n_init=1, max_iter=i)
    km.fit(X_blobs)
    
    sns.scatterplot(x=X_blobs[:, 0], y=X_blobs[:, 1], hue=km.labels_, palette='viridis', ax=ax, legend=False, alpha=0.6)
    ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1], s=200, c='red', marker='X', edgecolor='black')
    ax.set_title(f"Iteration {i}", fontsize=14)

plt.suptitle("The Expectation-Maximization (EM) steps of K-Means", fontsize=18)
plt.show()

### Why K-Means Fails on Stretched Data (And Why We Need GMM)
K-Means relies purely on Euclidean distance. It assumes every cluster is a perfect circle. If our data is stretched (anisotropic), K-Means will draw a completely illogical boundary.

**Gaussian Mixture Models (GMM)** fix this by calculating a Covariance Matrix, allowing the clusters to stretch into ellipses.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

sns.set_theme(style="whitegrid")

# 1. Generate tightly packed blobs (random_state=170 is the magic number here)
X_tight, _ = make_blobs(n_samples=600, random_state=170)

# 2. Stretch them into overlapping diagonal streaks
transformation = [[0.60834549, -0.63667341], [-0.40887718, 0.85253229]]
X_aniso = np.dot(X_tight, transformation)

# 3. Apply K-Means and GMM
km_stretched = KMeans(n_clusters=3, random_state=42).fit_predict(X_aniso)
gmm_stretched = GaussianMixture(n_components=3, covariance_type='full', random_state=42).fit_predict(X_aniso)

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

sns.scatterplot(x=X_aniso[:, 0], y=X_aniso[:, 1], hue=km_stretched, palette='viridis', ax=ax[0], legend=False)
ax[0].set_title("K-Means (Fails: Cuts perpendicularly based on distance)", fontsize=14)

sns.scatterplot(x=X_aniso[:, 0], y=X_aniso[:, 1], hue=gmm_stretched, palette='viridis', ax=ax[1], legend=False)
ax[1].set_title("GMM (Succeeds: Adapts to Covariance/Ellipses)", fontsize=14)
plt.show()

### The Geometry of Expectation-Maximization (EM)
By default, GMM converges incredibly fast because scikit-learn uses K-Means to initialize the starting positions! To see how the EM algorithm actually learns, we force it to initialize randomly.

Watch how the covariance matrices (represented by the contour lines) start as massive, confused circles and slowly tighten, rotate, and stretch into the optimal ovals.

In [ ]:
from IPython.display import clear_output
import time
from matplotlib.patches import Ellipse

X, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.60, random_state=0)
rng = np.random.RandomState(13)
X_stretched = np.dot(X, rng.randn(2, 2))

def draw_ellipse(position, covariance, ax=None, **kwargs):
    """Helper function to draw a 2D Gaussian ellipse"""
    ax = ax or plt.gca()
    # Convert covariance to principal axes
    if covariance.shape == (2, 2):
        U, s, Vt = np.linalg.svd(covariance)
        angle = np.degrees(np.arctan2(U[1, 0], U[0, 0]))
        width, height = 2 * np.sqrt(s)
    else:
        angle = 0
        width, height = 2 * np.sqrt(covariance)
    
    # Draw the Ellipse (multiplying by 2 roughly equals 2 standard deviations)
    for nsig in range(1, 3):
        ax.add_patch(Ellipse(position, nsig * width, nsig * height, angle=angle, **kwargs))

# --- THE LIVE MOVIE ---
# We will fit the GMM one step at a time (max_iter=1), passing the learned 
# parameters to the next step so we can watch it evolve.

n_components = 4
gmm = GaussianMixture(n_components=n_components, covariance_type='full', max_iter=1, warm_start=True)

plt.figure(figsize=(8, 6))

for i in range(15): # Watch 15 iterations
    clear_output(wait=True) # Clear the previous frame
    
    # Fit ONE step
    gmm.fit(X_stretched)
    labels = gmm.predict(X_stretched)
    
    # Plot the data
    plt.scatter(X_stretched[:, 0], X_stretched[:, 1], c=labels, cmap='viridis', zorder=2)
    
    # Plot the breathing ellipses
    for pos, covar, w in zip(gmm.means_, gmm.covariances_, gmm.weights_):
        draw_ellipse(pos, covar, alpha=w * 0.5, color='red', zorder=1)
        
    plt.title(f"Gaussian Mixture Model - Iteration {i+1}")
    plt.show()
    
    time.sleep(2) # Pause for half a second so we can see the frame
    
    if gmm.converged_:
        print(f"Converged at iteration {i+1}!")
        break

---
## Part 1.2: Non-Convex Clustering (DBSCAN & Spectral)

What happens when our data is shaped like interlocking moons? The concept of a "center" disappears entirely. Distance to a centroid becomes a useless metric. 

In [ ]:
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN

X_moons, y_moons = make_moons(n_samples=400, noise=0.08, random_state=42)

# Let's explore the Hyperparameter dependency of DBSCAN
# epsilon (eps) = Maximum distance between two samples to be considered neighbors
# min_samples = The number of neighbors required to become a "Core Point"

eps_values = [0.05, 0.15]
min_samples_values = [3, 10]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for i, eps in enumerate(eps_values):
    for j, min_s in enumerate(min_samples_values):
        db = DBSCAN(eps=eps, min_samples=min_s).fit_predict(X_moons)
        sns.scatterplot(x=X_moons[:,0], y=X_moons[:,1], hue=db, palette='tab10', ax=axes[i, j], legend=False)
        axes[i, j].set_title(f"eps={eps}, min_samples={min_s}")

plt.suptitle("DBSCAN: Highly dependent on Hyperparameters", fontsize=16)
plt.tight_layout()
plt.show()

### Spectral Clustering: The Magic of Eigenvectors
DBSCAN finds continuous density. **Spectral Clustering** turns the dataset into a Graph (nodes and edges), calculates the Graph Laplacian matrix, and looks at its Eigenvectors.

Watch what happens when we project the non-convex moons into the space of their first 2 eigenvectors. **The space physically untangles!**

In [ ]:
from sklearn.manifold import SpectralEmbedding
from sklearn.cluster import SpectralClustering

# 1. Compute the Eigenvector Embedding (The mathematical heart of Spectral Clustering)
embedder = SpectralEmbedding(n_components=2, affinity='nearest_neighbors', n_neighbors=10, random_state=42)
X_eigenvectors = embedder.fit_transform(X_moons)

# 2. Spectral Clustering is literally just running K-Means on these eigenvectors!
spectral_labels = SpectralClustering(n_clusters=2, affinity='nearest_neighbors', random_state=42).fit_predict(X_moons)

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
# Plot 1: The Raw Eigenvectors
sns.scatterplot(x=X_eigenvectors[:, 0], y=X_eigenvectors[:, 1], hue=spectral_labels, palette='Set1', ax=ax[0])
ax[0].set_title("The Eigenvector Space\n(Notice how the moons are now two distinct blobs!)", fontsize=14)
ax[0].set_xlabel("Eigenvector 1")
ax[0].set_ylabel("Eigenvector 2")

# Plot 2: The Final Labels mapped back to the original shapes
sns.scatterplot(x=X_moons[:, 0], y=X_moons[:, 1], hue=spectral_labels, palette='Set1', ax=ax[1])
ax[1].set_title("Final Spectral Clustering Result", fontsize=14)
plt.show()

---
## Part 2: Dimensionality Reduction (PCA vs t-SNE vs UMAP)

If your dataset has 64 features, K-Means is blind. The distance between points becomes mathematically meaningless (The Curse of Dimensionality). We must reduce the dimensions before we cluster.

Let's look at a famous 64-dimensional dataset: Images of handwritten digits. But first, how do we actually compress 64 dimensions into 2 dimensions while keeping the structural meaning intact? 

There are three major algorithmic paradigms in the industry.

### 1. Principal Component Analysis (PCA)
**The Philosophy:** "Smash the data linearly to maximize variance."
PCA is a linear projection algorithm. It looks for the angle (or axis) in the high-dimensional space where the data is most spread out. 

**How it works (The Math):**
1. **Center the data:** Subtract the mean from every feature.
2. **The Covariance Matrix:** Calculate the $n \times n$ covariance matrix $\Sigma = \frac{1}{n} X^T X$ to see how every feature correlates with every other feature.
3. **Eigen Decomposition:** Calculate the Eigenvectors and Eigenvalues of the covariance matrix. 
4. **Projection:** The Eigenvectors dictate the *direction* of the spread, and the Eigenvalues dictate the *magnitude* of that spread. PCA sorts the eigenvectors by their eigenvalues (from largest to smallest). The top 2 eigenvectors become your new 2D axes (Principal Components), and you project your dataset onto them using a dot product.

**The Verdict:** PCA is lightning fast, highly interpretable, and perfectly preserves **Global Structure**. However, because it is strictly linear, it completely fails to unfold complex, twisted geometric shapes (like a rolled-up piece of paper).

### 2. t-SNE (t-Distributed Stochastic Neighbor Embedding)
**The Philosophy:** "I don't care about the big picture, just keep my immediate neighbors close."
t-SNE abandons linear algebra for probability. It focuses purely on preserving **Local Neighborhoods**. If two points are close in 64D, they must be close in 2D. If they are far in 64D, t-SNE doesn't care exactly *how* far apart they are, as long as they are separated.

**How it works (The Math):**
1. **High-D Probabilities:** For every point, calculate a Gaussian probability distribution over all other points. If point $B$ is very close to point $A$, the probability $P_{A|B}$ is high.
2. **Low-D Probabilities:** Randomly scatter the points onto a 2D plane. Calculate the probabilities of these points being neighbors again, but this time use a **Student's t-distribution** (which has heavy tails). The heavy tails prevent the "Crowding Problem," ensuring points don't crush into a single black dot in the center.
3. **Gradient Descent:** Calculate the Kullback-Leibler (KL) Divergence (the difference between the High-D probabilities and Low-D probabilities). Move the points in the 2D space using gradient descent until the KL Divergence is minimized.

**The Verdict:** t-SNE produces incredibly beautiful, distinct visual clusters. However, it is extremely computationally slow, and it **destroys Global Structure**. The distance between two different clusters in a t-SNE plot is mathematically meaningless. Furthermore, it cannot be used to `transform()` new, unseen data in a pipeline.

### 3. UMAP (Uniform Manifold Approximation and Projection)
**The Philosophy:** "Use algebraic topology to preserve both local neighborhoods AND global structure."
Created in 2018, UMAP is the modern industry standard. It assumes the data is distributed on a topological manifold (a complex shape in high-dimensional space) and attempts to build a low-dimensional topological equivalent.

**How it works (The Math):**
1. **Simplicial Complex:** UMAP constructs a weighted graph (a 1-skeleton of a simplicial complex) connecting the data points. 
2. **Adaptive Local Connectivity:** Unlike t-SNE, which uses a fixed Gaussian variance, UMAP forces every point to be connected to at least its nearest neighbor. This means the concept of "distance" stretches and shrinks depending on how dense the data is in that specific region.
3. **Cross-Entropy Optimization:** It randomly initializes a 2D layout and uses Stochastic Gradient Descent to minimize the Cross-Entropy between the High-D graph and the Low-D graph. It balances attractive forces (pulling connected nodes together) and repulsive forces (pushing non-connected nodes apart).

**The Verdict:** UMAP is a masterpiece. It is significantly faster than t-SNE, it creates clusters that are just as beautiful, it preserves the global distances between clusters, and you can apply a fitted UMAP model to unseen test data.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
try:
    import umap
    has_umap = True
except ImportError:
    has_umap = False
    print("UMAP not installed. Run !pip install umap-learn to see the final plot.")

digits = load_digits()
X_digits, y_digits = digits.data, digits.target

# 1. PCA (Principal Component Analysis)
# Linear, fast, maximizes variance. Tries to preserve the GLOBAL structure.
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_digits)

# 2. t-SNE (t-Distributed Stochastic Neighbor Embedding)
# Non-linear, slow. Focuses purely on preserving LOCAL neighborhoods.
tsne = TSNE(n_components=3, random_state=42)
X_tsne = tsne.fit_transform(X_digits)

# Plotting
fig, axes = plt.subplots(1, 3 if has_umap else 2, figsize=(20 if has_umap else 14, 6))

sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=y_digits, palette='tab10', ax=axes[0], legend=False, s=15)
axes[0].set_title("PCA (Linear: Smashing Data Together)")

sns.scatterplot(x=X_tsne[:,0], y=X_tsne[:,1], hue=y_digits, palette='tab10', ax=axes[1], legend=False, s=15)
axes[1].set_title("t-SNE (Non-Linear: Beautiful Local Clusters)")

# 3. UMAP (Uniform Manifold Approximation and Projection)
# The modern industry standard. Preserves local AND global structure, and is incredibly fast.
if has_umap:
    reducer = umap.UMAP(random_state=42)
    X_umap = reducer.fit_transform(X_digits)
    sns.scatterplot(x=X_umap[:,0], y=X_umap[:,1], hue=y_digits, palette='tab10', ax=axes[2], legend=True, s=15)
    axes[2].set_title("UMAP (The Best of Both Worlds)")
    axes[2].legend(loc='center left', bbox_to_anchor=(1, 0.5), title='Digit')

plt.show()

## Part 3: Customer Segmentation

---
### Part 3.1: The Business Problem (Customer Segmentation)
You are the Lead Data Scientist for a major retail company. The marketing team has handed you a [dataset](https://www.kaggle.com/datasets/imakash3011/customer-personality-analysis) (`marketing_campaign.csv`) containing 29 features about their customers—their age, education, and how much they spend on wine, meat, fruits, and gold.

Marketing wants to run 3 or 4 highly targeted ad campaigns, but they don't know who their distinct customer groups are. Your job is to find them.

**Your Task:**
1. Load the dataset (Note: the Kaggle dataset is usually separated by tabs, so use `sep='\t'`).
2. Drop irrelevant ID columns and rows with missing data (`dropna()` is fine here to save time).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load data and clean NaNs


### Part 3.2: Preprocessing - The Unsupervised Trap
In supervised learning, if you forget to scale a Decision Tree, it still works. In unsupervised learning (PCA and K-Means), if you forget to scale your data, your entire model is garbage. PCA maximizes variance, so a feature measured in $100,000s will mathematically erase a feature measured in 1s.

**Your Task:**
1. Filter your dataframe so it ONLY contains numerical columns (PCA and standard K-Means cannot handle text categories like "PhD").
2. Apply `StandardScaler` to the resulting numeric dataframe.

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. Select only int64 and float64 columns

# 2. Scale the data


### Part 3.3: Dimensionality Reduction (PCA)
You have ~25 numeric features. Clustering in 25 dimensions suffers heavily from the **Curse of Dimensionality** (distances become meaningless). Let's compress the data.

**Your Task:**
1. Fit a `PCA()` model on your scaled data without specifying `n_components`.
2. Plot the `np.cumsum(pca.explained_variance_ratio_)`. This is called a Scree Plot.
3. Look at the plot: How many Principal Components do you need to retain 80% of the information (variance) in the dataset?
4. Create a new `pca_data` variable containing only that optimal number of components.

In [ ]:
from sklearn.decomposition import PCA

# YOUR PCA AND PLOTTING CODE HERE


### Part 3.4: Clustering & The Silhouette Score
Now we have a compressed dataset, free of collinearity. Let's find the customer groups.

**Your Task:**
1. Write a `for` loop that tests K-Means with `n_clusters` from 2 to 8.
2. Inside the loop, fit the model on `pca_data` and calculate the `silhouette_score(pca_data, labels)`.
3. Plot the Silhouette Scores. The peak of this graph is the mathematical optimal number of customer segments.
4. Retrain a final K-Means model using that optimal K.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# YOUR K-MEANS LOOP HERE


### Part 3.5: The "So What?" (Business Interpretation)
Math is useless if it doesn't drive business action. You have assigned every customer a cluster number (0, 1, 2, etc.), but what do those numbers actually *mean*?

**Your Task:**
1. Take the `labels_` array from your final K-Means model and add it as a new column called `Cluster` to your **ORIGINAL, unscaled dataframe**.
2. Use `df.groupby('Cluster').mean()` to look at the average values for each group.
3. Look at `Income`, `MntWines`, `MntMeatProducts`, and `NumDealsPurchases`.
4. Write down a "Persona Name" for each cluster (e.g., "Cluster 0: The High-Income Wine Snobs", "Cluster 1: The Low-Income Deal Hunters").

*The student who presents the clearest, most actionable personas to the instructor wins the practical!*

In [ ]:
# YOUR GROUPBY AND INTERPRETATION CODE HERE

---
## The Advanced Sandbox (Boss Mechanics)
Finished early? Try these advanced industry techniques:

**1. The Soft Clustering Reality (GMM):** K-Means forces a customer into exactly one cluster. But human behavior is messy. Implement a `GaussianMixture` model and use `predict_proba()` to find customers who sit exactly on the boundary between two personas (e.g., 50% Cluster A, 50% Cluster B).

**2. Beautiful Visualization (UMAP):** PCA is linear and boring. To make a beautiful presentation for the CEO, install UMAP (`!pip install umap-learn`). Project the data down to exactly 2 dimensions using `umap.UMAP()` and scatterplot it, coloring the points by your K-Means cluster labels. You will see islands of distinct customers appear!

In [ ]:
# YOUR BOSS MECHANIC CODE HERE